# 107 — Evaluación de fidelidad, cobertura y atribución

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** s1 "aprobado en 2015": implicada, [1] la respalda. s2 "dosis máxima
40 mg": implicada, [2] la respalda. s3 "no tiene efectos secundarios": NO implicada
(C no dice nada de efectos), y cita [2] que habla de dosis → cita falsa.
`faithfulness = 2/3 ≈ 0.67`; `citation recall = 2/3` (s3 sin cita válida);
`citation precision = 2/3 ≈ 0.67` (de las 3 citas emitidas, la de s3 no respalda su
afirmación). s3 es la más peligrosa: afirmación clínica inventada cuya cita le da
apariencia de procedencia — exactamente lo que la atribución rigurosa (AIS) detecta.

**Ejercicio 2.** Caso A: falla el **retriever** (no trae lo necesario; el generador es
fiel a un contexto incompleto). Primeras intervenciones: chunking (098), búsqueda
híbrida (100), multi-query o HyDE (103). Caso B: falla el **generador** (contexto
correcto, respuesta desviada). Intervenciones: prompt con regla de citas y rechazo
(102), re-ranking + umbral para limpiar ruido (101), modelo generador distinto.

**Ejercicio 3.** 1) Factuales directas con respuesta en un pasaje; 2) multi-hop que
cruzan documentos; 3) **sin respuesta en el corpus** (la respuesta correcta es "no
consta" — mide el rechazo); 4) adversariales/ambiguas (términos que aparecen en
documentos irrelevantes, versiones obsoletas de una política). Opcionalmente:
preguntas cuya respuesta cambió entre versiones del corpus, para probar invalidación.

**Ejercicio 4.** El contrato se verifica en el código: `kind == "evaluation"` y
`evidence` no vacía.

In [ ]:
result = run_lab("evaluation", seed=107)
assert result["kind"] == "evaluation"
assert result["evidence"]
show(result)


In [ ]:
claims = [
    ("aprobado en 2015", True, 1, True),
    ("dosis máxima 40 mg", True, 2, True),
    ("sin efectos secundarios", False, 2, False),
]

n = len(claims)
faithfulness = sum(1 for _, imp, _, _ in claims if imp) / n
citation_recall = sum(1 for _, _, _, ok in claims if ok) / n
citas_emitidas = [c for c in claims if c[2] is not None]
citation_precision = sum(1 for _, _, _, ok in citas_emitidas if ok) / len(citas_emitidas)

print("faithfulness       =", round(faithfulness, 3))
print("citation recall    =", round(citation_recall, 3))
print("citation precision =", round(citation_precision, 3))

diagnostico_a = "retriever: contexto incompleto → chunking / híbrida / multi-query"
diagnostico_b = "generador: se desvía del contexto → prompt con citas y rechazo / re-rank"
print("A:", diagnostico_a)
print("B:", diagnostico_b)

tipos_eval = ["factual directa", "multi-hop", "sin respuesta (rechazo)", "adversarial"]
print(tipos_eval)

## Reflexión

1. Construye (mentalmente) un caso con faithfulness = 1.0 y respuesta factualmente falsa. ¿Qué componente del pipeline falló y qué métrica lo habría detectado?
2. ¿Por qué faithfulness puede monitorearse sobre tráfico real de producción pero context recall no? ¿Qué se necesita para cada una?
3. Si tu juez LLM concuerda con humanos en el 78 % de los veredictos de entailment, ¿qué significa un cambio de faithfulness de 0.85 a 0.88 entre dos versiones del sistema?